# Model Usage
This notebook demonstrates how to use the trained image captioning model for generating captions, evaluating performance, and visualizing results.

In [ ]:
# Import required libraries for model usage and evaluation
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import pickle  # For loading saved objects

## Load Model and Tokenizer

In [ ]:
# Load trained model and supporting files
model = load_model('model_final.h5')  # Load the trained image captioning model
# Load tokenizer for text processing
with open('dataset/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)
# Load max_length for padding sequences
with open('dataset/max_length.pkl', 'rb') as f:
    max_length = pickle.load(f)
# Load image features for inference
with open('dataset/image_features.pkl', 'rb') as f:
    features = pickle.load(f)
# Load captions for evaluation
with open('dataset/captions.pkl', 'rb') as f:
    mapping = pickle.load(f)

## Generate Captions

In [ ]:
# Function to generate a caption for a given image using the trained model
# Iteratively predicts the next word until <end> token or max_length is reached

def predict_caption(model, image, tokenizer, max_length):
    in_text = '<start>'
    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)
        yhat = model.predict([image, sequence], verbose=0)
        yhat = np.argmax(yhat)
        word = tokenizer.index_word.get(yhat)
        if word is None:
            break
        in_text += ' ' + word
        if word == '<end>':
            break
    # Remove start/end tokens for final caption
    caption = in_text.replace('<start> ', '').replace(' <end>', '')
    return caption

## Evaluate the Model

In [ ]:
# Evaluate model performance using BLEU score
# Compares actual captions to predicted captions for each image
from nltk.translate.bleu_score import corpus_bleu
actual, predicted = [], []
for key in mapping.keys():
    feature = features[key]
    captions = mapping[key]
    actual_captions = [caption.split() for caption in captions]
    y_pred = predict_caption(model, feature, tokenizer, max_length)
    y_pred_words = y_pred.split()
    actual.append(actual_captions)
    predicted.append(y_pred_words)
bleu_score = corpus_bleu(actual, predicted)
print('BLEU score:', bleu_score)

## Visualize Results

In [ ]:
# Visualize actual and predicted captions for a given image
# Displays the image and prints both actual and predicted captions
from PIL import Image
import matplotlib.pyplot as plt

def visualize_caption(image_name):
    img_name = image_name.split('.')[0]
    img_path = f'dataset/{image_name}'
    image = Image.open(img_path)
    captions = mapping[img_name]
    print('Actual Captions:', captions)
    y_pred = predict_caption(model, features[img_name], tokenizer, max_length)
    print('Predicted Caption:', y_pred)
    plt.imshow(image)
    plt.title('Predicted Caption')
    plt.show()